# **●	Task1 — Problem & Dataset**

In [1]:
# Install required libraries
!pip install -q pyspark datasets pyarrow pandas

In [2]:
from datasets import load_dataset
from pyspark.sql import SparkSession
import pandas as pd
import os

In [3]:
spark = (
    SparkSession.builder
    .appName("Amazon Beauty Reviews Analysis")
    .getOrCreate()
)

print("Spark Version:", spark.version)


Spark Version: 4.0.3


In [5]:
# Load the Beauty & Personal Care category

dataset = load_dataset(
    "bagadbilla/amazon-reviews-2023-trimmed",
    data_dir="Beauty_and_Personal_Care",
    split="train"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/95 [00:00<?, ?it/s]

In [6]:
# 1. Save the full dataset in Hugging Face format (efficient)
dataset.save_to_disk('amazon_beauty_reviews_arrow')

# 2. Save a sample to CSV for portability
sample_to_save = dataset.select(range(10000))
sample_to_save.to_csv('amazon_beauty_sample.csv')

print("Dataset saved successfully!")

Saving the dataset (0/11 shards):   0%|          | 0/23911390 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Dataset saved successfully!


In [7]:
print(dataset)

Dataset({
    features: ['rating', 'title', 'text'],
    num_rows: 23911390
})


In [8]:
print("Total Records:", len(dataset))

Total Records: 23911390


In [9]:
print(dataset.features)

{'rating': Value('float64'), 'title': Value('string'), 'text': Value('string')}


In [10]:
dataset[0]

{'rating': 1.0,
 'title': 'Gasoline!! Seriously reeks of gasoline!',
 'text': 'Opened the package & instant migraine. I cannot believe the stench.  I have purchased other packages that did not smell at all so I do not know if these were a damaged shipment or damaged during packaging or what, but the minute I opened the Amazon package I smelled it before I even opened the Terra Tattoos package. I couldn’t believe it. Then I find that the pink inks from the back have smeared all over the fronts of the tattoos. Yes, you eventually take the clear part off to apply the tattoo, but I always lay it down with the clear covering first to line it up & I didn’t want to risk the pink ink transferring to my art projects so it’s going back. I’ll update the review when Amazon sends my replacement. I’m upset because now my resin is here & I don’t have the tattoos to do my project, but I’m more mad about the fact that I have a massive migraine because of the gasoline fumes that lingered.  I believe the

In [11]:
sample = dataset.select(range(1000))

pdf = sample.to_pandas()

pdf.head()

,rating,title,text
0,1.0,Gasoline!! Seriously reeks of gasoline!,Opened the package & instant migraine. I canno...
1,1.0,Useless! These have massive gaps at pump nozzle.,Tops the list for worst purchase. Tried these ...
2,5.0,Hailey loves unicorns!,Bought this for my granddaughter. Her entire ...
3,1.0,Not sharp enough to cut smoothly.,These are extremely dull and will wreck your n...
4,3.0,"Pretty, but didn’t close correctly.",Mine wouldn’t close correctly so I sent it bac...


In [12]:
print(pdf.columns.tolist())

['rating', 'title', 'text']


In [13]:
pdf.dtypes

,0
rating,float64
title,object
text,object


In [14]:
pdf.isnull().sum()

,0
rating,0
title,0
text,0


In [16]:
# Create a Spark DataFrame from a smaller sample (e.g., first 1000 records)
spark_df_sample = spark.createDataFrame(dataset.select(range(1000)))
spark_df_sample.printSchema()
spark_df_sample.show(5, truncate=False)
print("Total Rows in Sample Spark DataFrame:", spark_df_sample.count())

root
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- title: string (nullable = true)

+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [18]:
# Trying a slightly larger but manageable sample (100,000 rows)
spark_df = spark.createDataFrame(dataset.select(range(100000)))
print(f"Successfully created Spark DataFrame with {spark_df.count()} rows.")

Successfully created Spark DataFrame with 100000 rows.


In [19]:
spark_df.printSchema()

root
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- title: string (nullable = true)



In [20]:
spark_df.show(20, truncate=False)

+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [21]:
print("Total Rows:", spark_df.count())

Total Rows: 100000


In [22]:
print("Total Columns:", len(spark_df.columns))

Total Columns: 3


In [23]:
spark_df.dtypes

[('rating', 'double'), ('text', 'string'), ('title', 'string')]

In [24]:
spark_df.describe().show()

+-------+------------------+--------------------+--------------------+
|summary|            rating|                text|               title|
+-------+------------------+--------------------+--------------------+
|  count|            100000|              100000|              100000|
|   mean|           4.25604|                NULL|              3333.0|
| stddev|1.1511545378304797|                NULL|                NULL|
|    min|               1.0|                    |!!YES!! Highly em...|
|    max|               5.0|🤷🏼‍♀️ Smooth “r...|               🧖‍♀️|
+-------+------------------+--------------------+--------------------+



In [25]:
from pyspark.sql.functions import col, sum, when

spark_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in spark_df.columns
]).show()

+------+----+-----+
|rating|text|title|
+------+----+-----+
|     0|   0|    0|
+------+----+-----+



In [26]:
spark_df.groupBy("rating").count().orderBy("rating").show()

+------+-----+
|rating|count|
+------+-----+
|   1.0| 5694|
|   2.0| 4494|
|   3.0| 9087|
|   4.0|19964|
|   5.0|60761|
+------+-----+



In [27]:
print("Partitions:", spark_df.rdd.getNumPartitions())

Partitions: 2


In [28]:
spark_df.cache()

spark_df.count()

100000

In [29]:
import os

dataset_path = dataset.cache_files[0]["filename"]

size_gb = os.path.getsize(dataset_path) / (1024**3)

print(f"Dataset Size: {size_gb:.2f} GB")

Dataset Size: 0.47 GB
